# Concurrency, Idempotency, Reliability & Resilience — Design Addendum

**System:** File Validation & Routing Framework (`fvr`, MongoDB/GridFS + FastAPI) • **Version:** 1.0 • **Status:** DRAFT FOR REVIEW

## 1. Purpose & Scenario Catalog

Answers the operational questions: *concurrent uploads/workers, files failing in transfer, "upload succeeded" messages with no file in GridFS, and how the framework scales.* Every row maps a failure mode → prevention mechanism → detection → recovery:

| # | Failure mode | Prevented by | Detected by | Recovered by |
| --- | --- | --- | --- | --- |
| C1 | Same file uploaded twice (user double-click) | unique `file_registry.sha256_checksum` | 409/duplicate API response + `FILE_NOT_DUPLICATE` rule | user sees "already uploaded" with the original's status |
| C2 | Same event delivered twice (webhook replay) | unique `event_log (event_type, dedup_key)` | duplicate insert is a no-op | nothing to do — idempotent by construction |
| C3 | Two workers grab the same event | atomic claim `RECEIVED → PROCESSING` (findOneAndUpdate) | loser's claim matches 0 docs | loser moves on |
| C4 | Duplicate push to Intake volume | partial-unique `file_transfers (file_id, direction) WHERE COMPLETED` | transfer insert fails | second worker skips |
| C5 | Transfer to volume fails mid-copy | retry ladder + checksum verify on every attempt | `TRANSFER_FAILED` event + `status: RETRYING` | exponential backoff → auto-retry; GridFS copy retained, so re-transfer is always possible |
| C6 | User told "upload OK" but file never reached GridFS | **ack-after-durable-finalize contract** (below) | session completeness check (`declared vs received`) | session janitor flags INCOMPLETE; user resumes or re-uploads (sha256 dedup makes it safe) |
| C7 | Abandoned chunked upload (browser closed) | orphan-chunk janitor | session older than TTL with no finalize | chunks deleted, session marked ABANDONED, nothing routed |
| C8 | Validation crash on a "poison" file | per-file isolation + max retries | `FRAMEWORK_ERROR` count | after 3 retries → auto-quarantine with reason; pipeline keeps flowing |
| C9 | Volume/destination outage | circuit breaker + bounded queue | transfer failures spike, DLQ count | breaker opens → batch deferred → alert; reconciliation later proves `STUCK_IN_TRANSIT` |
| C10 | Worker crash mid-processing | change-stream resume token + at-least-once | token lag vs `received_at` | next worker resumes exactly at token; guarded state prevents double-apply |
| C11 | MongoDB transient failure | retryable writes + transaction retry on `TransientTransactionError` | driver error | auto-retry (idempotent writes) |
| C12 | Missed SharePoint webhook | watcher diff vs `file_registry` (self-healing sweep) | file present in source, absent in registry | sweep creates the `FILE_ARRIVED` event |

## 2. Concurrency Control Patterns (the write-pattern standard)

All FastAPI/worker writes follow **guarded atomic transitions** — never read-then-write:

```python
# PATTERN 1 — atomic claim (event worker, transfer worker)
doc = db.event_log.find_one_and_update(
    {"event_id": e, "processing_status": "RECEIVED"},        # guard
    {"$set": {"processing_status": "PROCESSING", "worker_id": me}},
    session=s)
if doc is None: return  # someone else claimed it — C3 resolved

# PATTERN 2 — guarded state transition (file_registry state machine)
res = db.file_registry.update_one(
    {"file_id": f, "current_status": "VALIDATING"},           # guard = expected state
    {"$set": {"current_status": "PASSED", "updated_at": now}})
if res.matched_count == 0: raise IllegalTransition(file_id, ...)  # audited

# PATTERN 3 — optimistic version for multi-field admin edits (config collections)
res = db.validation_rules.update_one(
    {"rule_id": r, "_v": expected_version},
    {"$set": {...}, "$inc": {"_v": 1}})
if res.matched_count == 0: raise StaleWrite → 409 to the admin UI

# PATTERN 4 — idempotency by unique index (structural, no code path needed)
#   sha256_checksum, (event_type, dedup_key), partial (file_id, direction) COMPLETED,
#   ingestion_id — duplicate writes fail at the DB, by design
```

| Concurrency surface | Control | Why it's safe |
| --- | --- | --- |
| N upload API pods behind LB | stateless app + all state in Mongo (no sticky sessions) | any pod serves any chunk/part |
| Parallel chunk uploads (one file) | GridFS `n`-indexed chunks + `$inc` part counters in `upload_sessions.parts[]` | order-independent; `$inc` is atomic |
| Competing event workers | Pattern 1 claim + change-stream resume tokens | work is partitioned, never shared |
| Concurrent re-classification vs validation | Pattern 2 guard | state machine rejects the late writer |
| Concurrent transfer attempts | Pattern 4 partial-unique + claim on `PENDING` | one winner |
| Purge vs new activity | purge transactions + guarded `gridfs_purged_at: null` update | no purge while any in-flight write matches |
| Multi-doc writes | MongoDB transactions (replica set) | all-or-nothing (purge, finalize, ingestion results) |

**Backpressure:** if `event_log` RECEIVED depth or `file_transfers` PENDING depth exceeds thresholds, API pods return `503 + Retry-After` on new uploads (protects the pipeline rather than silently queuing unbounded).

## 3. Deep-Dive A — File fails in transit (volume transfer)

```
transfer attempt ──✗──► status: RETRYING (attempts++) ──► backoff 30s/2m/10m/1h
     │                        │
     │ 5 consecutive failures │ success → dest_sha256 verified == source_sha256
     ▼                        ▼
status: FAILED + TRANSFER_FAILED event + notification (owner + ops)
     │
     ├─ auto: circuit-breaker check (dest outage? then all transfers defer, not just this one)
     └─ manual: ops re-drive from the board (one click re-queues PENDING)
```

**Why nothing is ever lost:** the GridFS copy is the system of record and is **never purged** until reconciliation says `MATCHED` — a failed transfer cannot destroy the source bytes. Every retry re-downloads from GridFS and re-verifies the destination checksum; `bytes_written` + `dest_sha256` prove the copy byte-for-byte.

**End-state guarantees:** reconciliation classifies any file with no COMPLETED transfer as `STUCK_IN_TRANSIT` (severity escalates with age) — there is no silent limbo; every failed transfer is either retried, re-driven, or visibly red on the ops board.

## 4. Deep-Dive B — "Upload succeeded" message but no file in GridFS

Root fix is a **contract**, not a retry:

> **The API acknowledges success ONLY after durable finalize** — i.e. after (a) the final chunk is written, (b) sha256 is computed, (c) `fs.files` is finalized with metadata, (d) the `file_registry` doc exists, and (e) the `FILE_ARRIVED` event is persisted — all in one transaction. If any step fails, the client gets 5xx + a resumable cursor, never "success".

Belt-and-braces for real-world divergence (proxies, client bugs):

1. **Client idempotency key** — every part carries `Idempotency-Key: session_id|filename|part_index` (same shape as `event_log.dedup_key`); safe client retries can never duplicate.
2. **Session completeness check** — `files_declared_count` (manifest) vs `files_received_count` (finalized registry docs); on `complete`, a mismatch returns **207 multi-status** listing per-file outcomes — the UI shows exactly which files landed and which didn't. No blanket "all good".
3. **Session janitor (scheduled, e.g. every 15 min)** — sessions `IN_PROGRESS` older than TTL (2 h proposed):
   * finalized files → kept, session marked `COMPLETED_PARTIAL`;
   * unfinalized parts with orphan GridFS chunks → chunks deleted, part marked `ABANDONED`;
   * user notified via portal + email: "upload incomplete — resume or re-upload" (re-upload is dedup-safe by sha256).
4. **Ops visibility** — INCOMPLETE/ABANDONED sessions are a first-class ops board filter (new session status enum).

**New `upload_sessions.status` enum:** `IN_PROGRESS → COMPLETED | COMPLETED_PARTIAL | INCOMPLETE | ABANDONED` (+ `files_received_count` authoritative from finalized parts only).

## 5. Poison-file handling (C8)

A file that crashes a validator (corrupt xlsx, zip bomb, encoding edge case) must not stall the queue:
* validator subprocess/timeouts isolated **per file** (one bad file never blocks others);
* `retry_count` max 3 on `FRAMEWORK_ERROR`;
* on exhaustion → `current_status: ROUTED_QUARANTINE` with `unidentified_reason: FRAMEWORK_ERROR_EXHAUSTED` + alert — the pipeline keeps flowing and a human investigates with the validation evidence attached;
* the failed validator's stack is captured in `validation_results.details` (truncated) for diagnosis.

## 6. Scalability Plan

| Axis | Mechanism | Trigger to scale |
| --- | --- | --- |
| Upload API | stateless FastAPI pods behind LB; chunked/resumable uploads | p95 upload latency SLO breach |
| Event workers | competing consumers on change streams (N workers, Pattern-1 claims partition work) | RECEIVED queue depth > 1k for 5 min |
| Transfer workers | independent pool claiming `PENDING` transfers | PENDING depth / STUCK_IN_TRANSIT age |
| Validation | per-file isolation → trivially parallel; FILE rules are O(1) metadata checks | throughput only |
| GridFS reads | parallel chunk reads; `secondaryPreferred` for retry scans | transfer throughput |
| MongoDB writes | replica set primary; bounded batches (500) everywhere | write contention (rare — writes are small + indexed) |
| Collections | indexes reviewed at R2; sharding only on triggers (HLD §8): `event_log` > ~50 GB retained, `file_registry` > ~50 M docs | capacity dashboard |
| Seasonal spikes | 503 backpressure protects correctness; queue drains at worker speed | — |

**The design scales by adding pods/workers, not by re-architecting** — all coordination is through atomic Mongo primitives (unique indexes, guarded updates, claims), which are O(1) regardless of worker count.

## 7. Idempotency Matrix (single reference)

| Operation | Idempotency key | Mechanism |
| --- | --- | --- |
| Event ingestion | `(event_type, dedup_key)` | unique index (C2) |
| File registration | `sha256_checksum` | unique index (C1) |
| Part/chunk upload | `session|file|part` (client key) + GridFS `(files_id, n)` | unique indexes |
| State transitions | expected-state guard | guarded update (C3, C10) |
| Volume transfer | partial-unique `(file_id, direction)` COMPLETED | unique index (C4) |
| Pushback ingestion | `ingestion_id` | upsert |
| Reconciliation verdicts | `(file_id, run_id)` | unique index |
| Purge | `gridfs_purged_at: null` guard | guarded update + transaction |
| Admin config edits | `_v` optimistic version | guarded update |

## 8. Resilience Layers (defense-in-depth, in firing order)

1. **Prevent** — unique indexes + guarded transitions (structural, not procedural).
2. **Retry** — exponential backoff on events, transfers, transient Mongo errors (retryable writes).
3. **Contain** — per-file isolation, poison-file auto-quarantine, circuit breakers, 503 backpressure, batch caps.
4. **Dead-letter** — `DEAD` events + `FAILED` transfers with re-drive UX; nothing silently dropped.
5. **Self-heal** — watcher diffs vs `file_registry` (missed webhooks), session janitor (ghost uploads).
6. **Prove** — reconciliation certifies the chain per file; `MATCHED` gates purging; every anomaly is a named, tracked status.
7. **Recover** — backups RPO ≤ 15 min / RTO ≤ 1 h (metadata + GridFS restored together).

**Golden rule:** every failure leaves a *named, queryable state* (never a limbo) and at least two independent detection paths (event/alert + reconciliation/sweep).

## 9. LLD Additions from this Addendum

| Change | Where |
| --- | --- |
| `upload_sessions.status` enum (IN_PROGRESS/COMPLETED/COMPLETED_PARTIAL/INCOMPLETE/ABANDONED) + index `{status, started_at}` | LLD §5.2 |
| Session janitor job (orphan-chunk cleanup, completeness finalization) | ops worker + `archive_jobs`-style control doc or dedicated `job` control |
| Client `Idempotency-Key` header contract | API spec (LLD §10 analog) |
| `_v` optimistic versioning on registry/config collections | LLD conventions |
| `503 + Retry-After` backpressure contract | API spec |
| Poison-file: `FRAMEWORK_ERROR` retry cap → auto-quarantine with `unidentified_reason: FRAMEWORK_ERROR_EXHAUSTED` | LLD state machine + FILE_STATUS usage |
| Circuit breaker state on transfer destination (in-memory, per worker pool + shared breaker doc) | ops runbook |
| Metrics: queue depths, DEAD count, STUCK_IN_TRANSIT age, INCOMPLETE sessions, orphan-chunk count | ops dashboard

## 10. Review Questions

1. Session TTL (2 h) and janitor cadence (15 min) — right defaults for your file sizes?
2. Backpressure thresholds — max queue depth before uploads get 503?
3. Poison-file retry cap (3) and auto-quarantine policy — acceptable?
4. Is `COMPLETED_PARTIAL` the right UX for multi-file sessions with mixed outcomes?
5. Circuit breaker thresholds for volume writes (failures/min, open duration)?